# Post-processing: standardise addresses

Cleans the extracted address values from the pipeline's long-format CSV
(`filename, entity_type, value, [confidence], error`):

1. expands common abbreviations to long form (`st` -> `street`, `rd` -> `road`, ...)
2. applies a **typo dictionary** you grow over time (`centnal` -> `central`)
3. tidies casing, comma spacing, and keeps UK postcodes upper-case

Uses only standard libraries — `pandas` + `re` / `difflib` / `collections`.
Edit the two dictionaries in the next cell; that's where all the project-specific
knowledge lives.

In [ ]:
import re
import difflib
from collections import Counter
import pandas as pd

## 1. The mappings (edit these)

- `ABBREVIATIONS`: stable shorthand -> long form.
- `TYPO_CORRECTIONS`: misspellings you spot in *your* data. Grow this list using
  the typo finder in section 4.

In [ ]:
ABBREVIATIONS = {
    # street / road types
    'rd': 'road', 'ave': 'avenue', 'av': 'avenue',
    'blvd': 'boulevard', 'ln': 'lane', 'dr': 'drive', 'ct': 'court',
    'pl': 'place', 'sq': 'square', 'ter': 'terrace', 'terr': 'terrace',
    'cres': 'crescent', 'cl': 'close', 'gdns': 'gardens', 'gdn': 'garden',
    'pk': 'park', 'hwy': 'highway', 'wlk': 'walk', 'grn': 'green',
    # unit / building
    'apt': 'apartment', 'appt': 'apartment', 'fl': 'flat', 'ste': 'suite',
    'bldg': 'building', 'mt': 'mount',
}

# Misspellings spotted in the data. Add to this as you go (see section 4).
TYPO_CORRECTIONS = {
    'centnal': 'central',
    'streer': 'street',
    'raod': 'road',
    'avneue': 'avenue',
    # '<typo>': '<correct>',
}

## 2. The standardisation function

Works token-by-token (case-insensitive), so abbreviations are only replaced as whole words,
never inside other words. All text is title-cased while preserving acronyms (NHS) and 
existing all-caps tokens.

In [ ]:
def _recase(word):
    """Title-case a word, but keep acronyms (NHS) and all-caps tokens as they are."""
    if not word:
        return word
    if word.isupper() and len(word) > 1:
        return word
    return word[0].upper() + word[1:].lower()


def standardise_address(text):
    """Standardise one address string. Returns non-strings/blank unchanged.

    Processes word-by-word, leaving all digits and punctuation (including hyphens
    inside place names like 'Stoke-on-Trent') exactly as they were.
    """
    # Extracted values may be NULL (NaN / None) or empty - pass those through
    # untouched (not stringified), so null stays null in the output.
    if not isinstance(text, str) or not text.strip():
        return text

    matches = list(re.finditer(r"[A-Za-z']+", text))  # the letter-words only
    out, last = [], 0
    for idx, m in enumerate(matches):
        start, end = m.span()
        out.append(text[last:start])  # keep the punctuation/digits between words verbatim
        word = m.group(0)
        low = word.lower()
        if low in TYPO_CORRECTIONS:
            w = TYPO_CORRECTIONS[low]
        elif low in ABBREVIATIONS:
            w = ABBREVIATIONS[low]
        else:
            w = word
        out.append(_recase(w))
        last = end
    out.append(text[last:])

    s = ''.join(out)
    s = re.sub(r"\s*,\s*", ', ', s)                       # tidy comma spacing
    s = re.sub(r"\b[A-Za-z]{1,2}\d[A-Za-z\d]?\s+\d[A-Za-z]{2}\b",
               lambda m: m.group(0).upper(), s)            # UK postcodes -> upper
    return s

## 4. Serial Number Processing

Extracts the last 6 digits from serial number entities. Non-numeric values, 
null entries, and serial numbers shorter than 6 digits are passed through unchanged.

In [ ]:
def extract_last_6_digits(value):
    """Extract the last 6 characters from a serial number after removing whitespace."""
    if not isinstance(value, str) or not value.strip():
        return value
    
    # Remove whitespace and return last 6 characters
    cleaned = value.replace(' ', '')
    return cleaned[-6:]

## 6. Date Standardisation

Standardises English dates to ISO format (YYYY-MM-DD). Handles:
- `dd/mm/yy` or `dd/mm/yyyy` (with or without leading zeros)
- Same patterns with `-` or space delimiters
- Long form: `1 January 2024`, `01 Jan 2024`, `January 1st, 2024`, etc.

Tracks unparseable dates for review so you can identify patterns to add.

In [ ]:
def standardise_date(value):
    """Standardise an English date to ISO format (YYYY-MM-DD).
    
    Handles: dd/mm/yy, dd/mm/yyyy, dd-mm-yy, dd mm yyyy, dot-delimited dates,
    and long form dates. Removes ordinal suffixes (st, nd, rd, th) before parsing.
    Returns non-strings/blank unchanged. Unparseable dates return None.
    """
    if not isinstance(value, str) or not value.strip():
        return value
    
    value = value.strip()
    
    # Remove ordinal suffixes (1st, 2nd, 3rd, 26th, 15TH, etc.)
    value = re.sub(r'(\d+)(st|nd|rd|th)\b', r'\1', value, flags=re.IGNORECASE)
    # Normalize apostrophes used before 2-digit years, e.g. "'69" -> "69"
    value = re.sub(r"(?<=\D)'(?=\d{2}\b)", '', value)
    # Normalize accidental spaces around dot separators, e.g. "01 . 02 . 2024"
    value = re.sub(r'\s*\.\s*', '.', value)
    
    # List of format strings to try (in order of specificity)
    formats = [
        '%d/%m/%Y',    # dd/mm/yyyy
        '%d/%m/%y',    # dd/mm/yy
        '%d-%m-%Y',    # dd-mm-yyyy
        '%d-%m-%y',    # dd-mm-yy
        '%d.%m.%Y',    # dd.mm.yyyy
        '%d.%m.%y',    # dd.mm.yy
        '%d %m %Y',    # dd mm yyyy
        '%d %m %y',    # dd mm yy
        '%d %B %Y',    # 1 January 2024
        '%d %B %y',    # 1 January 24
        '%d %b %Y',    # 1 Jan 2024
        '%d %b %y',    # 1 Jan 24
        '%B %d %Y',    # August 25 1969
        '%B %d, %Y',   # August 25, 1969
        '%B %d, %y',   # August 25, 69
        '%b %d %Y',    # Aug 25 1969
        '%b %d, %Y',   # Aug 25, 1969
        '%b %d, %y',   # Aug 25, 69
        '%d.%B.%Y',    # 1.January.2024
        '%d.%B.%y',    # 1.January.24
        '%d.%b.%Y',    # 1.Jan.2024
        '%d.%b.%y',    # 1.Jan.24
        '%B %d, %Y',   # January 1, 2024
        '%B %d, %y',   # January 1, 24
        '%b %d, %Y',   # Jan 1, 2024
        '%b %d, %y',   # Jan 1, 24
        '%d. %B %Y',   # 1. January 2024
        '%d. %b %Y',   # 1. Jan 2024
    ]
    
    # Try each format
    for fmt in formats:
        try:
            parse_value = value
            parse_fmt = fmt
            # Two-digit years in this dataset should resolve to the 1900s.
            if '%y' in fmt and '%Y' not in fmt:
                parse_value = re.sub(r'(?<=\D)(\d{2})(?=\D*$)', lambda m: f"19{m.group(1)}", value)
                parse_fmt = fmt.replace('%y', '%Y')
            parsed = pd.to_datetime(parse_value, format=parse_fmt)
            return parsed.strftime('%Y-%m-%d')
        except (ValueError, TypeError):
            continue
    
    # If all formats fail, return None for review
    return None

In [ ]:
# 8. Batch processing configuration and end-to-end export
# Each entry describes one input CSV and the entity types to process.
# Outputs are written to ./processed/<name>_processed.csv

DATASETS = [
    {
        'name': 'audiogram_p1_b1_e',
        'data': 'data/output/audiogram_p1_b1_e.csv',
        'address_entities': ['childs_address'],
        'date_entities': [],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'audiogram_p1_b1_l',
        'data': 'data/output/audiogram_p1_b1_l.csv',
        'address_entities': ['address'],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'educational_assessment_p1_b1_b',
        'data': 'data/output/educational_assessment_p1_b1_b.csv',
        'address_entities': [],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'educational_assessment_p1_b1_f',
        'data': 'data/output/educational_assessment_p1_b1_f.csv',
        'address_entities': [],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'medical_examinations_p1_b1_k',
        'data': 'data/output/medical_examinations_p1_b1_k.csv',
        'address_entities': ['present_address'],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'medical_questionnaire_p1_b1_d',
        'data': 'data/output/medical_questionnaire_p1_b1_d.csv',
        'address_entities': ['medical_examination_address'],
        'date_entities': ['medical_examination_date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'parental_interview_p1_b1_j',
        'data': 'data/output/parental_interview_p1_b1_j.csv',
        'address_entities': ['present_address', 'home_address_birth', 'place_of_birth'],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'parental_questionnaire_p1_b1_c',
        'data': 'data/output/parental_questionnaire_p1_b1_c.csv',
        'address_entities': ['present_address', 'place_of_birth', 'home_address_birth'],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
    {
        'name': 'perinatal_mortality_survey_p1_b1_a',
        'data': 'data/output/perinatal_mortality_survey_p1_b1_a.csv',
        'address_entities': ['place_of_residence', 'baby_delivered_address'],
        'date_entities': [],
        'code_entity': 'serial_number',
    },
    {
        'name': 'test_booklet_p1_b1_g',
        'data': 'data/output/test_booklet_p1_b1_g.csv',
        'address_entities': [],
        'date_entities': ['date'],
        'code_entity': 'childs_code_no',
    },
]

from pathlib import Path


def _null_or_blank(series):
    return series.isna() | series.astype(str).str.strip().eq('')


processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)


def process_dataset(dataset):
    df = pd.read_csv(dataset['data'], dtype=str)

    if 'file_name' in df.columns and 'filename' not in df.columns:
        df = df.rename(columns={'file_name': 'filename'})

    if 'serial_number_filename' not in df.columns:
        df['serial_number_filename'] = (
            df['filename']
            .astype(str)
            .str.split('_', n=1).str[0]
            .str.replace(r'[^0-9]', '', regex=True)
            .replace('', pd.NA)
        )

    address_entities = dataset.get('address_entities', [])
    date_entities = dataset.get('date_entities', [])
    df['value_clean'] = df['value']

    if address_entities:
        is_address = df['entity_type'].isin(address_entities)
        df.loc[is_address, 'value_clean'] = df.loc[is_address, 'value'].apply(standardise_address)

    if date_entities:
        is_date = df['entity_type'].isin(date_entities)
        df.loc[is_date, 'value_clean'] = df.loc[is_date, 'value'].apply(standardise_date)

    # Trim code entity values in long-format rows (this is where childs_code_no/serial_number exists).
    code_entity = dataset.get('code_entity', 'code')
    is_code = df['entity_type'].astype(str).eq(code_entity)
    if is_code.any():
        df.loc[is_code, 'value'] = df.loc[is_code, 'value'].apply(extract_last_6_digits)
        df.loc[is_code, 'value_clean'] = df.loc[is_code, 'value_clean'].apply(extract_last_6_digits)

    df_no_error = df.drop(columns=['error'], errors='ignore').copy()

    entity_nulls = (
        df_no_error
        .assign(
            raw_is_null=_null_or_blank(df_no_error['value']),
            clean_is_null=_null_or_blank(df_no_error['value_clean'])
        )
        .groupby('entity_type', dropna=False)
        .agg(
            rows=('entity_type', 'size'),
            raw_nulls=('raw_is_null', 'sum'),
            clean_nulls=('clean_is_null', 'sum')
        )
        .reset_index()
    )

    index_cols = ['filename', 'serial_number_filename']
    agg_first_non_null = lambda s: next((x for x in s if pd.notna(x)), s.iloc[0] if len(s) else None)

    value_wide = (
        df_no_error
        .pivot_table(
            index=index_cols,
            columns='entity_type',
            values=['value', 'value_clean'],
            aggfunc=agg_first_non_null
        )
    )

    value_wide.columns = [
        entity if metric == 'value' else f'{entity}_clean'
        for metric, entity in value_wide.columns
    ]
    value_wide = value_wide.reset_index()

    # Reorder columns so each original column is immediately followed by its _clean pair.
    entity_order = df_no_error['entity_type'].dropna().astype(str).drop_duplicates().tolist()
    ordered_entity_cols = []
    for ent in entity_order:
        if ent in value_wide.columns:
            ordered_entity_cols.append(ent)
        clean_col = f'{ent}_clean'
        if clean_col in value_wide.columns:
            ordered_entity_cols.append(clean_col)

    leading_cols = [c for c in ['filename', 'serial_number_filename'] if c in value_wide.columns]
    trailing_cols = [c for c in value_wide.columns if c not in (leading_cols + ordered_entity_cols)]
    value_wide = value_wide[leading_cols + ordered_entity_cols + trailing_cols]

    if code_entity in value_wide.columns:
        filename_digits = value_wide['serial_number_filename'].astype(str).str.replace(r'[^0-9]', '', regex=True)
        entity_digits = value_wide[code_entity].astype(str).str.replace(r'[^0-9]', '', regex=True)
        filename_digits = filename_digits.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
        entity_digits = entity_digits.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
        value_wide['serial_number_match_flag'] = (
            filename_digits.notna() & entity_digits.notna() & (filename_digits == entity_digits)
        )
    else:
        value_wide['serial_number_match_flag'] = pd.NA

    metrics = {
        'rows_out': len(value_wide),
        'code_entity': code_entity,
        'code_rows': int(is_code.sum()),
        'code_nulls_after_trim': int((is_code & _null_or_blank(df['value'])).sum()),
        'serial_match_true': int(value_wide['serial_number_match_flag'].eq(True).sum()),
        'serial_match_false': int(value_wide['serial_number_match_flag'].eq(False).sum()),
        'serial_match_na': int(value_wide['serial_number_match_flag'].isna().sum()),
    }

    output_path = processed_dir / f"{dataset['name']}_processed.csv"
    value_wide.to_csv(output_path, index=False)
    return output_path, value_wide, metrics, entity_nulls


results = []
entity_null_logs = []

for dataset in DATASETS:
    output_path, preview, metrics, entity_nulls = process_dataset(dataset)

    print(
        f"[{dataset['name']}] rows={metrics['rows_out']} "
        f"code_entity={metrics['code_entity']} code_rows={metrics['code_rows']} "
        f"code_nulls_after_trim={metrics['code_nulls_after_trim']} "
        f"serial_match_true={metrics['serial_match_true']} "
        f"serial_match_false={metrics['serial_match_false']} "
        f"serial_match_na={metrics['serial_match_na']}"
    )

    entity_nulls_view = entity_nulls[['entity_type', 'rows', 'raw_nulls', 'clean_nulls']].copy()
    print(entity_nulls_view.to_string(index=False))
    print('')

    entity_nulls['name'] = dataset['name']
    entity_null_logs.append(entity_nulls)

    results.append({
        'name': dataset['name'],
        'output_path': str(output_path),
        'rows': metrics['rows_out'],
        'code_entity': metrics['code_entity'],
        'code_rows': metrics['code_rows'],
        'code_nulls_after_trim': metrics['code_nulls_after_trim'],
        'serial_match_true': metrics['serial_match_true'],
        'serial_match_false': metrics['serial_match_false'],
        'serial_match_na': metrics['serial_match_na'],
        'status': 'processed',
    })

results_df = pd.DataFrame(results)
entity_nulls_df = pd.concat(entity_null_logs, ignore_index=True) if entity_null_logs else pd.DataFrame()

results_df